# Session 3 — Evaluate and interpret one lymph-node prediction

## Goal

Connect DGAT output to germinal-center biology using marker maps, predicted-protein clustering, global Moran's I, and pairwise bivariate Moran's I. **Every protein value in this notebook is inferred, not measured.** Spatial coherence and co-localization support hypotheses but do not demonstrate accuracy, direct interaction, signaling, or causality.


In [ ]:
REQUIREMENTS = [('anndata', 'anndata==0.11.4'), ('scanpy', 'scanpy==1.11.5'), ('seaborn', 'seaborn==0.13.2'), ('igraph', 'igraph==0.11.8'), ('leidenalg', 'leidenalg==0.10.2'), ('networkx', 'networkx==3.4.2')]

from pathlib import Path
import importlib, importlib.util, json, os, shutil, subprocess, sys

SESSION_REQUIREMENTS = REQUIREMENTS
in_colab = importlib.util.find_spec("google.colab") is not None and Path("/content").is_dir()
if in_colab:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    repo_dir = Path("/content/ECCB-2026-Tutorial")
    tutorial_root = repo_dir / "hands-on_tutorial"
    if not (tutorial_root / "src" / "dgat_tutorial").is_dir():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/osmanbeyoglulab/ECCB-2026-Tutorial.git", str(repo_dir)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo_dir), "fetch", "--depth", "1", "origin", "main"], check=True)
        subprocess.run(["git", "-C", str(repo_dir), "reset", "--hard", "origin/main"], check=True)
    drive_root = Path("/content/drive/MyDrive/ECCB2026")
    manifest_path = drive_root / "asset_manifest.json"
    if not manifest_path.is_file(): raise FileNotFoundError(f"Missing {manifest_path}; run Session 0.")
    manifest = json.loads(manifest_path.read_text())
    drive_assets = drive_root / "assets" / "DGAT_assets"
    local_assets = tutorial_root / "external" / "DGAT_assets"
    local_data = local_assets / "data"
    local_data.mkdir(parents=True, exist_ok=True)
    for filename in ("V1_Human_Lymph_Node_filtered_feature_bc_matrix.h5", "V1_Human_Lymph_Node_manual_GC_annot.csv"):
        source, destination = drive_assets / "data" / filename, local_data / filename
        expected = manifest["files"][filename]["bytes"]
        if not source.is_file() or source.stat().st_size != expected: raise IOError(f"Invalid Drive asset: {source}")
        if not destination.is_file() or destination.stat().st_size != expected: shutil.copy2(source, destination)
    source_spatial, destination_spatial = drive_assets / "data" / "spatial", local_data / "spatial"
    if not source_spatial.is_dir(): raise FileNotFoundError(f"Missing {source_spatial}")
    if destination_spatial.exists(): shutil.rmtree(destination_spatial)
    shutil.copytree(source_spatial, destination_spatial)
    os.environ["DGAT_TUTORIAL_STATE_DIR"] = str(drive_root / "state")
    dgat_dir = tutorial_root / "external" / "DGAT"
    if not (dgat_dir / "utils" / "Preprocessing.py").is_file():
        dgat_dir.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/osmanbeyoglulab/DGAT.git", str(dgat_dir)], check=True)
    missing = [spec for module, spec in SESSION_REQUIREMENTS if importlib.util.find_spec(module) is None]
    if missing:
        wheelhouse = drive_root / "wheelhouse" / f"py{sys.version_info.major}{sys.version_info.minor}"
        cmd = [sys.executable, "-m", "pip", "install", "-q", *missing]
        if wheelhouse.is_dir(): cmd[4:4] = ["--find-links", str(wheelhouse)]
        subprocess.run(cmd, check=True); importlib.invalidate_caches()
else:
    tutorial_root = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "src" / "dgat_tutorial").is_dir()), None)
    if tutorial_root is None: raise FileNotFoundError("Could not locate hands-on_tutorial.")
    drive_root = None

os.chdir(tutorial_root)
sys.path.insert(0, str(tutorial_root / "src"))
from dgat_tutorial.checkpoints import tutorial_paths, write_checkpoint
paths = tutorial_paths(tutorial_root)
print("Tutorial root:", paths.root)
print("Completed checkpoints:", sorted(p.name for p in paths.checkpoints.glob("session_*/part_*.json")) or "none")


## Step 1 — Load one aligned workflow(estimated_runtime 26sec)

This cell proves that RNA, coordinates, GC labels, and the validated precomputed predictions refer to the same ordered spots. It also checks the organizer provenance before analysis.

In [ ]:
import json, matplotlib.pyplot as plt, networkx as nx, numpy as np, pandas as pd, scanpy as sc, seaborn as sns
from scipy.cluster.hierarchy import leaves_list, linkage
from scipy.spatial.distance import squareform
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from dgat_tutorial.checkpoints import preferred_prediction_path
from dgat_tutorial.dgat import load_prediction_metadata, load_prediction_table
from dgat_tutorial.alignment import require_exact_identifiers
from dgat_tutorial.plotting import plot_spatial_feature
from dgat_tutorial.processing import normalize_total_log1p
from dgat_tutorial.evaluation import symmetric_knn_weights, morans_i_from_weights, bivariate_morans_i_from_weights

spots = pd.read_csv(paths.processed_data/"filtered_spots.csv",index_col=0)
rna_counts = pd.read_csv(paths.processed_data/"corresponding_rna_counts.csv",index_col=0)
gc = pd.read_csv(paths.processed_data/"germinal_center_labels.csv",index_col=0).iloc[:,0].astype(str).eq("True")
prediction_path = preferred_prediction_path(paths); predicted = load_prediction_table(prediction_path); metadata = load_prediction_metadata(prediction_path)
for name,index in [("RNA",rna_counts.index),("GC",gc.index),("predictions",predicted.index)]: require_exact_identifiers(spots.index,index,left_name="Session 1 spots",right_name=name)
if metadata.get("dataset") != "V1_Human_Lymph_Node": raise ValueError("Prediction sidecar names a different sample")
predicted = predicted.loc[spots.index]; rna_counts = rna_counts.loc[spots.index]; gc = gc.loc[spots.index]
markers=["CD19","CXCR5","PDCD1"]
for marker in markers:
    if marker not in rna_counts or marker not in predicted: raise KeyError(f"Missing {marker}")
evaluation_message = metadata.get("evaluation_scope") or metadata.get("evaluation_note")
if not evaluation_message: raise ValueError("Prediction metadata does not state its evaluation scope")
print(evaluation_message); print(f"Aligned {len(spots)} spots, {predicted.shape[1]} predicted proteins, {int(gc.sum())} GC spots")


## Step 2 — Compare canonical GC mRNA and predicted protein maps(estimated_runtime 26sec)

CD19 marks B-cell identity, CXCR5 supports follicular homing, and PDCD1 encodes PD-1. Compare *locations*, not colorbar magnitudes: RNA and predicted protein use different transformations and units. Discordance may reflect biology, coverage, or model error.


In [ ]:
rna_log = normalize_total_log1p(rna_counts)
fig,axes=plt.subplots(3,2,figsize=(9,12))
for row,marker in enumerate(markers):
    label="PD-1 (PDCD1)" if marker=="PDCD1" else marker
    plot_spatial_feature(spots,rna_log[marker],f"{label} mRNA",ax=axes[row,0],cmap="viridis")
    plot_spatial_feature(spots,predicted[marker],f"{label} predicted protein",ax=axes[row,1],cmap="magma")
marker_path=paths.figures/"session03_gc_marker_rna_vs_predicted.png"; fig.tight_layout(); fig.savefig(marker_path,dpi=170); plt.show()


## Step 3 — Cluster spots using all predicted proteins(estimated_runtime 40sec)

Standardization prevents high-variance proteins from dominating. PCA denoises the 31-protein profile; Leiden then finds communities in a 10-neighbor graph at the paper's lymph-node resolution (`0.25`). The fixed seed makes the teaching result reproducible. Cluster IDs are not cell types.


In [ ]:
scaled=(predicted-predicted.mean())/predicted.std(ddof=0).replace(0,1)
n_pcs=min(20,scaled.shape[1],scaled.shape[0]-1); pcs=PCA(n_components=n_pcs,random_state=7).fit_transform(scaled)
cluster_adata=sc.AnnData(pcs); cluster_adata.obs_names=spots.index
sc.pp.neighbors(cluster_adata,n_neighbors=10,use_rep="X",random_state=7)
sc.tl.leiden(cluster_adata,resolution=0.25,random_state=7,key_added="predicted_protein_cluster",flavor="igraph",n_iterations=2)
clusters=cluster_adata.obs["predicted_protein_cluster"].astype(str).loc[spots.index]
fig,axes=plt.subplots(1,2,figsize=(11,4.8))
cluster_codes=pd.Categorical(clusters).codes
plot_spatial_feature(spots,pd.Series(cluster_codes,index=spots.index),"Leiden clusters: predicted proteins",ax=axes[0],cmap="tab20",add_colorbar=False)
plot_spatial_feature(spots,gc.astype(int),"Manual germinal-center annotation",ax=axes[1],cmap="coolwarm",vmin=0,vmax=1)
cluster_path=paths.figures/"session03_clusters_vs_gc.png"; fig.tight_layout(); fig.savefig(cluster_path,dpi=170); plt.show()


## Step 4 — Quantify cluster overlap with germinal centers(estimated_runtime 1sec)

Each cluster is treated in turn as the positive GC prediction. The best F1 cluster is reported transparently. Because selection and scoring use the same sample, this is descriptive anatomical concordance—not independent validation.


In [ ]:
rows=[]
for cluster in sorted(clusters.unique(),key=int):
    calls=clusters.eq(cluster)
    rows.append({"cluster":cluster,"spots":int(calls.sum()),"gc_overlap":int((calls&gc).sum()),"accuracy":accuracy_score(gc,calls),"precision":precision_score(gc,calls,zero_division=0),"recall":recall_score(gc,calls,zero_division=0),"f1":f1_score(gc,calls,zero_division=0)})
cluster_metrics=pd.DataFrame(rows).sort_values("f1",ascending=False); best_cluster=str(cluster_metrics.iloc[0]["cluster"])
contingency=pd.crosstab(clusters.rename("cluster"),gc.map({False:"Other",True:"GC"}).rename("annotation"))
metrics_path=paths.results/"session03_cluster_gc_metrics.csv"; contingency_path=paths.results/"session03_cluster_gc_contingency.csv"
cluster_metrics.to_csv(metrics_path,index=False); contingency.to_csv(contingency_path)
print("Best descriptive GC-overlap cluster:",best_cluster); display(cluster_metrics,contingency)
if contingency.to_numpy().sum()!=len(spots): raise AssertionError("Contingency total mismatch")


## Step 5 — Compare univariate Moran's I(estimated_runtime 5sec)

Global Moran's I asks whether similar values occur in neighboring spots. We use one symmetric six-nearest-neighbor graph for every marker and modality. Higher predicted-protein Moran's I may reflect recovered structure **or model smoothing**; it is not proof of protein accuracy.


In [ ]:
weights=symmetric_knn_weights(spots,n_neighbors=6)
rows=[]
for protein in predicted.columns:
    gene=protein.split("_")[0]
    if gene in rna_log.columns:
        rows.append({"protein":protein,"gene":gene,"predicted_protein_morans_i":morans_i_from_weights(predicted[protein],weights),"mrna_morans_i":morans_i_from_weights(rna_log[gene],weights)})
moran_table=pd.DataFrame(rows).dropna(); moran_table["difference"]=moran_table["predicted_protein_morans_i"]-moran_table["mrna_morans_i"]
moran_path=paths.results/"session03_predicted_vs_mrna_morans_i.csv"; moran_table.to_csv(moran_path,index=False)
fig,ax=plt.subplots(figsize=(6,5)); sns.scatterplot(data=moran_table,x="mrna_morans_i",y="predicted_protein_morans_i",ax=ax)
lo=min(ax.get_xlim()[0],ax.get_ylim()[0]); hi=max(ax.get_xlim()[1],ax.get_ylim()[1]); ax.plot([lo,hi],[lo,hi],"k--",lw=1)
for marker in markers: 
    row=moran_table[moran_table.protein.eq(marker)]
    if len(row): ax.text(row.mrna_morans_i.iloc[0],row.predicted_protein_morans_i.iloc[0],"PD-1" if marker=="PDCD1" else marker)
ax.set_title("Spatial coherence: corresponding mRNA vs predicted protein")
moran_fig=paths.figures/"session03_morans_i_comparison.png"; fig.tight_layout(); fig.savefig(moran_fig,dpi=170); plt.show(); display(moran_table.sort_values("difference",ascending=False))


## Step 6 — Bivariate Moran's I heatmap and network(estimated_runtime 53sec)

Bivariate Moran's I measures whether high values of one predicted protein occur near high values of another. The heatmap summarizes all pairs; the network retains `I ≥ 0.70`, matching Figure 5f–g's high-confidence visualization rule. These edges indicate spatial co-localization only—not binding, signaling, or direct biological interaction.


In [ ]:
proteins=list(predicted.columns); matrix=pd.DataFrame(np.eye(len(proteins)),index=proteins,columns=proteins,dtype=float)
for i,a in enumerate(proteins):
    for j in range(i+1,len(proteins)):
        b=proteins[j]; value=bivariate_morans_i_from_weights(predicted[a],predicted[b],weights); matrix.loc[a,b]=matrix.loc[b,a]=value
if not np.allclose(matrix,matrix.T,equal_nan=True): raise AssertionError("Bivariate matrix is not symmetric")
bivariate_path=paths.results/"session03_bivariate_morans_i.csv"; matrix.to_csv(bivariate_path)
distance=1-matrix.clip(-1,1); np.fill_diagonal(distance.values,0); order=list(matrix.index[leaves_list(linkage(squareform(distance.values,checks=False),method="average"))])
ordered=matrix.loc[order,order]; mask=np.triu(np.ones_like(ordered,dtype=bool),k=1)
fig,axes=plt.subplots(1,2,figsize=(15,7),gridspec_kw={"width_ratios":[1.15,1]})
sns.heatmap(ordered,mask=mask,cmap="coolwarm",center=0,vmin=-1,vmax=1,square=True,ax=axes[0],cbar_kws={"label":"Bivariate Moran's I"}); axes[0].set_title("Predicted-protein spatial co-localization")
threshold=.70; graph=nx.Graph(); graph.add_nodes_from(proteins)
for i,a in enumerate(proteins):
    for b in proteins[i+1:]:
        value=float(matrix.loc[a,b])
        if value>=threshold: graph.add_edge(a,b,weight=value)
pos=nx.spring_layout(graph,seed=7,weight="weight"); widths=[1+5*(d["weight"]-threshold)/(1-threshold) for *_,d in graph.edges(data=True)]
nx.draw_networkx(graph,pos,ax=axes[1],node_size=500,font_size=7,width=widths,node_color="#79b7d3",edge_color="#d95f02",alpha=.85); axes[1].set_title(f"Spatial association network (I ≥ {threshold:.2f})"); axes[1].axis("off")
bivariate_fig=paths.figures/"session03_bivariate_heatmap_network.png"; fig.tight_layout(); fig.savefig(bivariate_fig,dpi=180,bbox_inches="tight"); plt.show()
edge_table=pd.DataFrame([{"protein_a":a,"protein_b":b,"bivariate_morans_i":d["weight"]} for a,b,d in graph.edges(data=True)]).sort_values("bivariate_morans_i",ascending=False) if graph.number_of_edges() else pd.DataFrame(columns=["protein_a","protein_b","bivariate_morans_i"])
edges_path=paths.results/"session03_bivariate_network_edges.csv"; edge_table.to_csv(edges_path,index=False); display(edge_table.head(15))


## Step 7 — Save the interpretation handoff(estimated_runtime 1sec)

The saved prompts keep inference limits attached to the figures when results are discussed after the workshop.


In [ ]:
prompts="""# Lymph-node interpretation prompts\n\n- Do CD19, CXCR5, and PD-1 predictions concentrate in or around annotated GCs?\n- Could low detected-gene coverage explain any discordant region?\n- Which clusters overlap GCs, and how was the best cluster selected?\n- Could stronger predicted-protein Moran's I reflect smoothing?\n- Which bivariate edges are plausible co-localization hypotheses, and what orthogonal experiment would test them?\n\nInferred proteins are predictions. Spatial co-localization does not establish direct interaction or causality.\n"""
prompt_path=paths.results/"session03_interpretation_prompts.md"; prompt_path.write_text(prompts)
manifest=write_checkpoint("3.1",[marker_path,cluster_path,metrics_path,contingency_path,moran_path,moran_fig,bivariate_path,bivariate_fig,edges_path,prompt_path],summary={"sample":"V1_Human_Lymph_Node","spots":len(spots),"clusters":int(clusters.nunique()),"best_gc_cluster":best_cluster,"best_f1":float(cluster_metrics.iloc[0].f1),"network_threshold":.70,"network_edges":graph.number_of_edges()},start=paths.root)
print(prompts); print("Checkpoint:",manifest)


## Checks and limitations

- Re-run from the top and confirm the same Leiden labels and metrics.
- Constant signals should return undefined Moran's I; shuffled signals should usually approach zero; constructed spatial gradients should be positive.
- Low RNA coverage can degrade inference.
- Moran results depend on the six-neighbor graph, and smoothing can increase them.
- GC overlap is descriptive because the best cluster is selected and scored on this sample.
- Multi-method benchmarking, ablation, cross-tissue validation, and immunofluorescence validation are covered in the lecture rather than reproduced here.


## Next steps

Use these results to form biological hypotheses, then seek measured proteins or orthogonal imaging for validation. The hands-on workflow is complete: one sample from input QC through DGAT inference and spatial interpretation.
